# Analysis of experiment #10: CFCP instances for geometric regions

## Read data

In [1]:
# Read csv

import pandas as pd
import numpy as np

df = pd.read_csv("stats10.csv")

# Remove final _ in solver name
df['solver'] = df['solver'].str.rstrip('_')

# Choose solvers to keep
df = df[df['solver'].isin(['byp', 'gurobi'])]

# Change solver name
df['solver'] = df['solver'].replace('byp', 'b&p')

# Add column with the density of the graph
df["density"] = df.apply(lambda row: 2 * row.nedges / (row.nvertices * (row.nvertices - 1)), axis=1)

# Add column with the number of centers and the radious
df["centers"] = df.instance.apply(lambda s: int(s.split("_")[1][1:]))
df["radius"] = df.instance.apply(lambda s: int(s.split("_")[2][1:]))
df["i"] = df.instance.apply(lambda s: int(s.split("_")[3][1:]))

# Format number of nodes for cplex
df.loc[(df.solver == 'cplex') & (df.nodes == 0), 'nodes'] = 1

# Unify TIME_EXCEEDED values
df = df.replace({'TIME_EXCEEDED_PR': 'TIME_EXCEEDED', 'TIME_EXCEEDED_LP': 'TIME_EXCEEDED'}) 

# Add column for each possible status
df["isOptimal"] = df.state == "OPTIMAL"
df["isInfeasible"] = df.state == "INFEASIBLE"
df["isFeasible"] = df.state == "TIME_EXCEEDED"

# Replace gap equal to -1 with NaN
df['gap'] = df['gap'].replace(-1, np.nan)

# Absolute gap
df['absGap'] = df.apply(lambda row: row.ub - row.lb if not np.isnan(row.gap) else np.nan, axis=1)

# Remove the heuristic time from the total time
df.time = df.time - df.initialHeurTime

# Format timelimit
df.loc[df.state == 'TIME_EXCEEDED', 'time'] = 1200

df = df.sort_values(by=["instance", "solver"]).reset_index(drop=True)
print(list(df.columns))
df.head()

['instance', 'solver', 'run', 'nvertices', 'nedges', 'nP', 'nQ', 'nvars', 'ncons', 'state', 'terminationReason', 'time', 'nodes', 'nodesLeft', 'lb', 'ub', 'gap', 'initialHeurValue', 'initialHeurTime', 'initialSemigreedyIters', 'nNodesInt', 'nNodesFrac', 'nNodesGcp', 'nNodesTrivial', 'nNodesInfeas', 'nNodesInfeasPrepro', 'nNodesInfeasCheck', 'nNodesInfeasAux', 'gcpAvgTime', 'nsol', 'nsolHeur', 'nsolLR', 'nsolGCP', 'nsolTrivial', 'ninitSol', 'ninitDummy', 'ninit', 'rootNVertices', 'rootNEdges', 'rootNP', 'rootNQ', 'rootlb', 'rootub', 'rootHeurTime', 'rootFeasTime', 'rootCgTime', 'rootNCalls', 'rootNCallsPool', 'rootNCallsHeur', 'rootNCallsMwis1', 'rootNCallsMwis2', 'rootNCallsExact', 'rootNCols', 'rootNColsPool', 'rootNColsHeur', 'rootNColsMwis1', 'rootNColsMwis2', 'rootNColsExact', 'rootTime', 'rootTimePool', 'rootTimeHeur', 'rootTimeMwis1', 'rootTimeMwis2', 'rootTimeExact', 'otherNodesHeurTime', 'otherNodesFeasNCalls', 'otherNodesFeasTime', 'otherNodesNCalls', 'otherNodesNCallsPool', '

,instance,solver,run,nvertices,nedges,nP,nQ,nvars,ncons,state,...,otherNodesTimeMwis2,otherNodesTimeExact,density,centers,radius,i,isOptimal,isInfeasible,isFeasible,absGap
0,circle_n12_r35_i0,b&p,0,570,105243,111,12,-1,-1,OPTIMAL,...,0.500883,0.059965,0.648987,12,35,0,True,False,False,0.0
1,circle_n12_r35_i0,gurobi,0,570,105243,111,12,2855,1098726,TIME_EXCEEDED,...,0.000000,0.000000,0.648987,12,35,0,False,False,True,2.0
2,circle_n12_r35_i1,b&p,0,229,11854,68,12,-1,-1,OPTIMAL,...,0.025688,0.017682,0.454072,12,35,1,True,False,False,0.0
3,circle_n12_r35_i1,gurobi,0,229,11854,68,12,920,108276,OPTIMAL,...,0.000000,0.000000,0.454072,12,35,1,True,False,False,0.0
4,circle_n12_r35_i2,b&p,0,337,31068,85,12,-1,-1,OPTIMAL,...,0.090149,0.004414,0.548749,12,35,2,True,False,False,0.0


In [2]:
df_avg = df.groupby(["centers","radius","solver"]).agg(
    {
        "nvertices": "mean",
        "density": "mean",
        "nP": "mean",
        "nQ": "mean",
        "isOptimal": "sum",
        "isInfeasible": "sum",
        "isFeasible": "sum",
        "time": "mean",
        "absGap": "mean",
    }
).reset_index()

columns = [("c", ""), ("r", ""), ("solver", ""), ("|V|", ""), ("density", ""), ("n", ""), ("m", ""),
           ("# status", "opt"), ("# status", "infeas"), ("# status", "tilim"), ("time (s)",""), ("abs gap", "")]
df_avg.columns = pd.MultiIndex.from_tuples(columns)
df_avg[("time (s)", "")] = df_avg[("time (s)", "")].astype(int)
df_avg[("abs gap", "")] = df_avg[("abs gap", "")].round(2)
df_avg[("|V|", "")] = df_avg[("|V|", "")].astype(int)
df_avg[("density", "")] = df_avg[("density", "")].round(2)
df_avg[("n", "")] = df_avg[("n", "")].astype(int)
df_avg[("m", "")] = df_avg[("m", "")].astype(int)
df_avg = df_avg.sort_values(by=[("r", ""),("c", "")]).reset_index(drop=True)
df_avg

c   r  solver  |V| density   n   m # status              time (s) abs gap
                                            opt infeas tilim                 
0  18  20     b&p  250    0.29  90  18        1      0     4      964     0.8
1  18  20  gurobi  250    0.29  90  18        5      0     0      147     0.0
2  12  35     b&p  370    0.54  88  12        4      0     1      301     0.2
3  12  35  gurobi  370    0.54  88  12        3      0     2      699     0.6
4   9  50     b&p  256    0.70  57   9        5      0     0       39     0.0
5   9  50  gurobi  256    0.70  57   9        5      0     0      344     0.0

In [3]:
# pivot table by solver
df2 = df.pivot_table(index=["instance", "centers", "radius", "nvertices", "density", "nP", "nQ", "initialHeurValue"], 
                     columns="solver", 
                     values=["time", "nodes", "lb", "ub"],
                     ).reset_index()
print(list(df2.columns))
df2.head()

[('instance', ''), ('centers', ''), ('radius', ''), ('nvertices', ''), ('density', ''), ('nP', ''), ('nQ', ''), ('initialHeurValue', ''), ('lb', 'b&p'), ('lb', 'gurobi'), ('nodes', 'b&p'), ('nodes', 'gurobi'), ('time', 'b&p'), ('time', 'gurobi'), ('ub', 'b&p'), ('ub', 'gurobi')]


instance centers radius nvertices   density   nP  nQ  \
solver                                                                  
0       circle_n12_r35_i0      12     35       570  0.648987  111  12   
1       circle_n12_r35_i1      12     35       229  0.454072   68  12   
2       circle_n12_r35_i2      12     35       337  0.548749   85  12   
3       circle_n12_r35_i3      12     35       449  0.582495  104  12   
4       circle_n12_r35_i4      12     35       269  0.459690   75  12   

       initialHeurValue   lb          nodes             time             ub  \
solver                   b&p gurobi     b&p gurobi       b&p    gurobi  b&p   
0                   5.0  3.0    2.0   187.0    1.0   231.323  1200.000  3.0   
1                   4.0  3.0    3.0   261.0  253.0    22.231   110.165  3.0   
2                   4.0  3.0    3.0   117.0    1.0    27.887   656.290  3.0   
3                   4.0  3.0    2.0  2076.0    1.0  1200.000  1200.000  4.0   
4                   4.0  3.0    3.0   183.0  904.0    24.561   330.281  3.0   

               
solver gurobi  
0         4.0  
1         3.0  
2         3.0  
3         3.0  
4         3.0

In [4]:
df3 = df2[[('instance',''), ('centers',''), ('radius',''), ('nvertices',''), ('density',''), ('nP',''), ('nQ',''), ('initialHeurValue',''), 
           ('time','b&p'), ('time','gurobi'), ('nodes','b&p'), ('nodes','gurobi'), 
           ('lb','b&p'), ('lb','gurobi'), ('ub','b&p'), ('ub','gurobi')]]
colNames = pd.MultiIndex.from_tuples([('instance',''), ('c',''), ('r',''), ('|V|',''), ('density',''), ('n',''), ('m',''), ('hval',''), 
           ('time (s)','b&p'), ('time (s)','gurobi'), ('nodes','b&p'), ('nodes','gurobi'), 
           ('lb','b&p'), ('lb','gurobi'), ('ub','b&p'), ('ub','gurobi')])
df3.columns = colNames
df3[('density','')] = df3[('density','')].round(2)
df3[('hval','')] = df3[('hval','')].astype(int)
df3[('nodes','b&p')] = df3[('nodes','b&p')].astype(int)
df3[('nodes','gurobi')] = df3[('nodes','gurobi')].astype(int)
df3[('lb','b&p')] = df3[('lb','b&p')].round(2)
df3[('lb','gurobi')] = df3[('lb','gurobi')].round(2)
df3[('ub','b&p')] = df3[('ub','b&p')].astype(int)
df3[('ub','gurobi')] = df3[('ub','gurobi')].astype(int)

# Specific formatting
df3[('time (s)','b&p')] = df3[('time (s)','b&p')].apply(lambda x: "{:.1f}".format(x) if x < 1200 else "tilim")
df3[('time (s)','gurobi')] = df3[('time (s)','gurobi')].apply(lambda x: "{:.1f}".format(x) if x < 1200 else "tilim")

df3 = df3.sort_values(by=('r','')).reset_index(drop=True)
df3

instance   c   r  |V| density    n   m hval time (s)         \
                                                              b&p gurobi   
0   circle_n18_r20_i1  18  20  212    0.25   86  18    4     24.9   86.4   
1   circle_n18_r20_i2  18  20  288    0.37   96  18    4    tilim  214.1   
2   circle_n18_r20_i0  18  20  257    0.22   94  18    4    tilim  324.2   
3   circle_n18_r20_i3  18  20  231    0.27   87  18    4    tilim   60.6   
4   circle_n18_r20_i4  18  20  265    0.33   91  18    4    tilim   51.9   
5   circle_n12_r35_i2  12  35  337    0.55   85  12    4     27.9  656.3   
6   circle_n12_r35_i3  12  35  449    0.58  104  12    4    tilim  tilim   
7   circle_n12_r35_i0  12  35  570    0.65  111  12    5    231.3  tilim   
8   circle_n12_r35_i4  12  35  269    0.46   75  12    4     24.6  330.3   
9   circle_n12_r35_i1  12  35  229    0.45   68  12    4     22.2  110.2   
10   circle_n9_r50_i0   9  50  297    0.72   62   9    3      0.7  707.3   
11   circle_n9_r50_i1   9  50  244    0.68   56   9    3    185.7  168.6   
12   circle_n9_r50_i2   9  50  205    0.69   49   9    4      6.7  223.5   
13   circle_n9_r50_i3   9  50  276    0.70   62   9    3      0.6  429.0   
14   circle_n9_r50_i4   9  50  258    0.71   56   9    3      2.7  194.9   

    nodes          lb         ub         
      b&p gurobi  b&p gurobi b&p gurobi  
0     503      3  3.0    3.0   3      3  
1    9166    585  3.0    3.0   4      3  
2   22799   2553  3.0    4.0   4      4  
3   15707      1  3.0    3.0   4      3  
4   15044      1  3.0    3.0   4      3  
5     117      1  3.0    3.0   3      3  
6    2076      1  3.0    2.0   4      3  
7     187      1  3.0    2.0   3      4  
8     183    904  3.0    3.0   3      3  
9     261    253  3.0    3.0   3      3  
10      3      1  3.0    3.0   3      3  
11   1265    124  3.0    3.0   3      3  
12    103     46  3.0    3.0   3      3  
13      3     36  3.0    3.0   3      3  
14     21    522  3.0    3.0   3      3

In [5]:
df3.to_latex("table10.tex", index=False, float_format="%.1f", na_rep="--")